# NLP Pipeline — Text Classification from Scratch

**Dataset:** 20 Newsgroups (sklearn built-in — zero setup, no Kaggle auth)  
**Mirrors:** Any Kaggle text classification task  

| Track | Approach | Time to first submission |
|---|---|---|
| Baseline | TF-IDF + Logistic Regression | ~10 min |
| Upgrade | BiLSTM from scratch (PyTorch) | ~45 min |

**Rule:** No pretrained models. Everything built from tokens up.

## Stage 0 — Load Data

In [1]:
import re
import numpy as np
from collections import Counter
from sklearn.datasets import fetch_20newsgroups

# 4 categories keeps training fast on CPU.
# On Kaggle day: load train.csv / test.csv instead.
CATEGORIES = ['sci.space', 'rec.sport.hockey', 'talk.politics.guns', 'comp.graphics']

raw_train = fetch_20newsgroups(subset='train', categories=CATEGORIES,
                               remove=('headers', 'footers', 'quotes'))
raw_test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES,
                               remove=('headers', 'footers', 'quotes'))

texts_train, y_train = raw_train.data, np.array(raw_train.target)
texts_test,  y_test  = raw_test.data,  np.array(raw_test.target)
label_names           = raw_train.target_names

print(f'Train samples : {len(texts_train)}')
print(f'Test  samples : {len(texts_test)}')
print(f'Classes       : {label_names}')
print(f'Class dist    : {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'\nSample text:\n{texts_train[0][:300]}')

Train samples : 2323
Test  samples : 1546
Classes       : ['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.politics.guns']
Class dist    : {0: 584, 1: 600, 2: 593, 3: 546}

Sample text:

Agreed here...I'll never forget Dan Kelly calling the play-by-play in the '87
Canada Cup.  He was masterful!

And Danny Gallivan will _never_ be replaced; even now when I watch HNIC I
remember his voice...when I see an Al MacInnis or Al Iafrate (hey, what's with
these guys named Al who can shoot??)


## Stage 1 — Text Cleaning

Lower-case, strip punctuation, collapse whitespace.  
On competition day you may also want to:
- Remove stopwords (for TF-IDF, not for LSTM)
- Handle HTML tags if data is from the web
- Keep numbers or replace with `<NUM>` token

In [2]:
def clean(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # keep alphanumeric + spaces
    text = re.sub(r'\s+', ' ', text).strip()   # collapse whitespace
    return text

texts_train_clean = [clean(t) for t in texts_train]
texts_test_clean  = [clean(t) for t in texts_test]

print('BEFORE:', texts_train[0][:150])
print()
print('AFTER :', texts_train_clean[0][:150])

BEFORE: 
Agreed here...I'll never forget Dan Kelly calling the play-by-play in the '87
Canada Cup.  He was masterful!

And Danny Gallivan will _never_ be repl

AFTER : agreed here i ll never forget dan kelly calling the play by play in the 87 canada cup he was masterful and danny gallivan will never be replaced even 


---
## Stage 2A — TF-IDF + Logistic Regression (Baseline)

### Why TF-IDF?

**TF** (Term Frequency) = how often a word appears in *this* document  
**IDF** (Inverse Document Frequency) = `log(N / df)` — penalizes words that appear in *every* document ("the", "is", etc.)

Result: a sparse vector that emphasizes words *specific* to this document.

We stack **two** vectorizers:
- `word (1,2)-grams` — captures meaning at word level
- `char (2,4)-grams` — catches morphology, typos, compound words

In [3]:
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Word-level TF-IDF
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2),    # unigrams + bigrams
    max_features=50_000,   # top 50k by corpus frequency
    sublinear_tf=True,     # use log(1+tf) -- dampens very frequent terms
    min_df=2,              # ignore terms in < 2 docs
)

# Character-level TF-IDF
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',    # char n-grams within word boundaries
    ngram_range=(2, 4),
    max_features=30_000,
    sublinear_tf=True,
    min_df=2,
)

X_word_tr = tfidf_word.fit_transform(texts_train_clean)
X_char_tr = tfidf_char.fit_transform(texts_train_clean)
X_word_te = tfidf_word.transform(texts_test_clean)     # ONLY transform, never fit on test!
X_char_te = tfidf_char.transform(texts_test_clean)

# Stack horizontally -> single feature matrix
X_tr = sp.hstack([X_word_tr, X_char_tr])
X_te = sp.hstack([X_word_te, X_char_te])

print(f'Train matrix shape : {X_tr.shape}  (samples x features)')
print(f'Sparsity           : {1 - X_tr.nnz / np.prod(X_tr.shape):.4f}  (most entries = 0)')

Train matrix shape : (2323, 80000)  (samples x features)
Sparsity           : 0.9874  (most entries = 0)


In [4]:
logreg = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
logreg.fit(X_tr, y_train)

preds_tfidf = logreg.predict(X_te)
acc_tfidf   = accuracy_score(y_test, preds_tfidf)

print(f'Test Accuracy (TF-IDF + LR): {acc_tfidf:.4f}\n')
print(classification_report(y_test, preds_tfidf, target_names=label_names))

Test Accuracy (TF-IDF + LR): 0.8875

                    precision    recall  f1-score   support

     comp.graphics       0.91      0.90      0.91       389
  rec.sport.hockey       0.89      0.94      0.91       399
         sci.space       0.85      0.85      0.85       394
talk.politics.guns       0.91      0.85      0.88       364

          accuracy                           0.89      1546
         macro avg       0.89      0.89      0.89      1546
      weighted avg       0.89      0.89      0.89      1546



In [5]:
# --- What words drive each class? ---
feature_names = tfidf_word.get_feature_names_out()
print('Top 10 words per class:')
for cls_idx, cls_name in enumerate(label_names):
    coef = logreg.coef_[cls_idx][:len(feature_names)]
    top  = np.argsort(coef)[-10:][::-1]
    print(f'  {cls_name}: {list(feature_names[top])}')

Top 10 words per class:
  comp.graphics: ['graphics', 'image', 'file', 'computer', '3d', 'hi', 'files', 'card', 'images', 'hello']
  rec.sport.hockey: ['hockey', 'team', 'game', 'he', 'games', 'season', 'nhl', 'play', 'players', 'teams']
  sci.space: ['space', 'nasa', 'orbit', 'moon', 'launch', 'earth', 'the moon', 'shuttle', 'solar', 'sci']
  talk.politics.guns: ['gun', 'guns', 'weapons', 'fbi', 'law', 'com', 'fire', 'no', 'people', 'were']


---
## Stage 2B — BiLSTM from Scratch (PyTorch)

### Architecture

```
Token indices  (B, L)
     |
Embedding      (B, L, E)   -- learned lookup table: token -> dense vector
     |
Dropout
     |
Bi-LSTM        (B, L, 2H)  -- forward LSTM + backward LSTM running in parallel
     |                        take LAST hidden state of each direction
concat(hn[-2], hn[-1])     (B, 2H)
     |
Dropout
     |
Linear         (B, n_classes)  -- raw logits
     |
CrossEntropyLoss  (includes softmax internally)
```

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN    = 256
EMBED_DIM  = 128
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.3
BATCH_SIZE = 64
EPOCHS     = 5
LR         = 1e-3

print(f'Device: {DEVICE}')

Device: cpu


In [7]:
# --- Build Vocabulary ---
# We map each unique word -> integer index.
# <PAD> = 0  (used to fill shorter sequences to MAX_LEN)
# <UNK> = 1  (for words not seen during training)

def tokenize(text: str):
    return text.split()

def build_vocab(texts, min_freq=2, max_vocab=30_000):
    counter = Counter(tok for t in texts for tok in tokenize(t))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.most_common(max_vocab):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab      = build_vocab(texts_train_clean)
VOCAB_SIZE = len(vocab)
PAD_IDX    = 0

print(f'Vocabulary size: {VOCAB_SIZE:,} tokens')
print(f'(words appearing < 2x mapped to <UNK>)')

# Show what encoding looks like
sample = texts_train_clean[0]
toks   = tokenize(sample)[:10]
ids    = [vocab.get(t, 1) for t in toks]
print(f'\nSample tokens : {toks}')
print(f'Encoded IDs   : {ids}')

Vocabulary size: 16,705 tokens
(words appearing < 2x mapped to <UNK>)

Sample tokens : ['agreed', 'here', 'i', 'll', 'never', 'forget', 'dan', 'kelly', 'calling', 'the']
Encoded IDs   : [2296, 128, 8, 194, 269, 1541, 1991, 2905, 2121, 2]


In [8]:
def encode(text: str, vocab: dict, max_len: int = MAX_LEN):
    toks = tokenize(text)[:max_len]           # truncate if too long
    ids  = [vocab.get(t, 1) for t in toks]    # unknown -> 1
    pad  = [PAD_IDX] * (max_len - len(ids))   # right-pad with 0
    return ids + pad


class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.X = [encode(t, vocab, max_len) for t in texts]
        self.y = labels

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return (torch.tensor(self.X[i], dtype=torch.long),
                torch.tensor(self.y[i],  dtype=torch.long))


le       = LabelEncoder()
y_tr_enc = le.fit_transform(y_train)
y_te_enc = le.transform(y_test)
N_CLASSES = len(le.classes_)

train_ds     = TextDataset(texts_train_clean, y_tr_enc, vocab)
test_ds      = TextDataset(texts_test_clean,  y_te_enc, vocab)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
xb, yb = next(iter(train_loader))
print(f'Batch X shape: {xb.shape}  -> (batch, seq_len)')
print(f'Batch y shape: {yb.shape}')
print(f'Padding check (last 5 of first sample): {xb[0, -5:].tolist()}')

Batch X shape: torch.Size([64, 256])  -> (batch, seq_len)
Batch y shape: torch.Size([64])
Padding check (last 5 of first sample): [0, 0, 0, 0, 0]


In [9]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 n_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()

        # Embedding: integer -> dense vector (learned during training)
        # padding_idx=0 means <PAD> tokens contribute zero gradient
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Bidirectional LSTM:
        #   forward  pass: reads sequence left  -> right
        #   backward pass: reads sequence right -> left
        #   output hidden dim = hidden_dim * 2
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
        )

        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim * 2, n_classes)

    def forward(self, x):
        # x: (B, L) token indices
        emb = self.drop(self.embed(x))           # (B, L, E)

        # out: hidden state at every timestep     (B, L, 2H)
        # hn : final hidden states                (2*n_layers, B, H)
        out, (hn, cn) = self.lstm(emb)

        # hn[-2] = last layer, forward direction  (B, H)
        # hn[-1] = last layer, backward direction (B, H)
        h = torch.cat([hn[-2], hn[-1]], dim=1)   # (B, 2H)

        return self.fc(self.drop(h))             # (B, n_classes) -- raw logits


model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM, n_classes=N_CLASSES,
    n_layers=N_LAYERS, dropout=DROPOUT,
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

# Quick shape check
with torch.no_grad():
    dummy_out = model(xb.to(DEVICE))
    print(f'Forward check: input {xb.shape} -> output {dummy_out.shape}  (should be [64, {N_CLASSES}])')

BiLSTMClassifier(
  (embed): Embedding(16705, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=4, bias=True)
)

Trainable parameters: 4,507,780


Forward check: input torch.Size([64, 256]) -> output torch.Size([64, 4])  (should be [64, 4])


## Training Loop

- **Loss:** `CrossEntropyLoss` = `-log(softmax(logit)[true_class])`
- **Optimizer:** Adam with `lr=1e-3`
- **Backprop:** `.backward()` computes gradients, `.step()` updates weights

In [10]:
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

print('{:>6} | {:>10} | {:>9} | {:>7}'.format('Epoch', 'Train Loss', 'Train Acc', 'Val Acc'))
print('-' * 42)

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = correct = total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()          # clear gradients from last step
        logits = model(xb)             # forward pass
        loss   = criterion(logits, yb) # compute loss
        loss.backward()                # backprop
        optimizer.step()               # update weights

        total_loss += loss.item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += len(yb)

    train_loss = total_loss / total
    train_acc  = correct / total

    # --- Validate ---
    model.eval()
    val_correct = val_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            val_correct += (model(xb).argmax(1) == yb).sum().item()
            val_total   += len(yb)
    val_acc = val_correct / val_total

    print('{:>6} | {:>10.4f} | {:>9.4f} | {:>7.4f}'.format(epoch, train_loss, train_acc, val_acc))

 Epoch | Train Loss | Train Acc | Val Acc
------------------------------------------


     1 |     1.3433 |    0.3353 |  0.3881


     2 |     1.2129 |    0.4529 |  0.4366


     3 |     1.1137 |    0.5110 |  0.4767


     4 |     0.9832 |    0.5781 |  0.4398


     5 |     0.9512 |    0.5949 |  0.5155


## Inference + Final Comparison

In [11]:
model.eval()
all_preds = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(DEVICE)
        # softmax converts raw logits -> probabilities summing to 1
        probs = torch.softmax(model(xb), dim=1)
        all_preds.extend(probs.argmax(1).cpu().numpy())

all_preds = np.array(all_preds)
acc_lstm  = accuracy_score(y_te_enc, all_preds)

print(f'Test Accuracy (BiLSTM): {acc_lstm:.4f}\n')
print(classification_report(y_te_enc, all_preds, target_names=label_names))

Test Accuracy (BiLSTM): 0.5155

                    precision    recall  f1-score   support

     comp.graphics       0.65      0.77      0.71       389
  rec.sport.hockey       0.54      0.59      0.56       399
         sci.space       0.37      0.44      0.40       394
talk.politics.guns       0.48      0.25      0.33       364

          accuracy                           0.52      1546
         macro avg       0.51      0.51      0.50      1546
      weighted avg       0.51      0.52      0.50      1546



In [12]:
print('=' * 40)
print('SUMMARY')
print('=' * 40)
print(f'TF-IDF + LogReg : {acc_tfidf:.4f}')
print(f'BiLSTM (scratch): {acc_lstm:.4f}')
print()
print('Competition strategy:')
print('  1. Submit TF-IDF first (~10 min) to get on the board.')
print('  2. Train LSTM in parallel, submit when done.')
print('  3. Ensemble: 0.4*tfidf_probs + 0.6*lstm_probs for extra +1-2%.')

SUMMARY
TF-IDF + LogReg : 0.8875
BiLSTM (scratch): 0.5155

Competition strategy:
  1. Submit TF-IDF first (~10 min) to get on the board.
  2. Train LSTM in parallel, submit when done.
  3. Ensemble: 0.4*tfidf_probs + 0.6*lstm_probs for extra +1-2%.


---
## Kaggle Day: Swap-In Checklist

```python
# Replace these lines at the top:
train = pd.read_csv('/kaggle/input/.../train.csv')
test  = pd.read_csv('/kaggle/input/.../test.csv')
sub   = pd.read_csv('/kaggle/input/.../sample_submission.csv')

TEXT_COL = 'text'   # actual column name
TARGET   = 'label'  # actual target column

texts_train_clean = [clean(t) for t in train[TEXT_COL]]
texts_test_clean  = [clean(t) for t in test[TEXT_COL]]
y_train = train[TARGET]

# At the end:
sub[TARGET] = le.inverse_transform(all_preds)  # if labels were strings
sub.to_csv('submission.csv', index=False)
```